# Diabetic Retinopathy Model Training

This notebook contains the complete training process for the dual-channel diabetic retinopathy detection model. The training logic from `train.py` has been divided into logical blocks so you can check each step individually.

## Training Process Overview

1. **Setup Python Path**: Add project root to Python path for local imports (essential for Paperspace)
2. **Setup and Configuration**: Import libraries and set up paths
3. **CUDA/GPU Check**: Verify GPU availability and configure TensorFlow
4. **Training Configuration**: Set up hyperparameters and configuration options
5. **Model Creation**: Create the dual-channel model architecture
6. **Dataset Loading**: Load and prepare the EyePACS datasets
7. **Optimizer Setup**: Configure the Adam optimizer with hyperparameters
8. **Callbacks Setup**: Set up training callbacks (checkpointing, early stopping, LR reduction)
9. **Model Compilation**: Compile the model with optimizer, loss, and metrics
10. **Training**: Train the model on the training dataset
11. **Evaluation**: Evaluate the model on test dataset
12. **Visualization**: Plot training history and metrics

## Running this Notebook

This notebook is designed to run on Paperspace or other remote Jupyter servers. 

**Before running**, make sure to install dependencies:
```bash
pip install -r requirements.txt
```

Then simply run the notebook cells in order.


In [ ]:
# Setup Python path for local project imports
# This is essential for Paperspace/remote servers where the project is not installed as a package

import sys
from pathlib import Path

# Known project paths (for Paperspace)
KNOWN_PATHS = [
    Path("/notebooks/sam-ai"),  # Paperspace path
]

# Get the current notebook's directory
notebook_dir = Path().resolve()

# Try to find project root by looking for pyproject.toml
project_root = None

# First, check known paths (for Paperspace)
for known_path in KNOWN_PATHS:
    if known_path.exists() and (known_path / "pyproject.toml").exists():
        project_root = known_path
        print(f"✓ Found project root using known path: {project_root}")
        break

# If not found in known paths, try relative detection
if project_root is None:
    # The notebook is in notebooks/, so project root is one level up
    project_root = notebook_dir.parent
    
    # Verify we found the project root by checking for pyproject.toml
    if not (project_root / "pyproject.toml").exists():
        # If not found, try current directory (in case notebook is run from project root)
        if (notebook_dir / "pyproject.toml").exists():
            project_root = notebook_dir
        else:
            # Last resort: try going up one more level
            project_root = project_root.parent
            if not (project_root / "pyproject.toml").exists():
                # Final check: try known paths even if they don't exist yet (for first run)
                for known_path in KNOWN_PATHS:
                    if (known_path / "pyproject.toml").exists():
                        project_root = known_path
                        break
                
                if project_root is None or not (project_root / "pyproject.toml").exists():
                    raise FileNotFoundError(
                        f"Could not find project root (pyproject.toml).\n"
                        f"Checked locations:\n"
                        f"  - {notebook_dir.parent}\n"
                        f"  - {notebook_dir}\n"
                        f"  - {notebook_dir.parent.parent}\n"
                        f"  - Known paths: {KNOWN_PATHS}\n"
                        f"Current working directory: {Path.cwd()}\n"
                        f"Notebook directory: {notebook_dir}\n"
                        f"Make sure you're running this notebook from within the project directory."
                    )

# Add project root to Python path if not already there
project_root = project_root.resolve()  # Resolve to absolute path
project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)
    print(f"✓ Added project root to Python path: {project_root}")
else:
    print(f"✓ Project root already in Python path: {project_root}")

# Verify sam_ml can be imported
try:
    import sam_ml
    print(f"✓ Successfully imported sam_ml from: {sam_ml.__file__}")
except ImportError as e:
    raise ImportError(
        f"Failed to import sam_ml. Project root: {project_root}\n"
        f"Make sure the sam_ml directory exists in the project root.\n"
        f"Sam_ml path should be: {project_root / 'sam_ml'}\n"
        f"Error: {e}"
    )

print(f"\nProject structure verified:")
print(f"  Project root: {project_root}")
print(f"  Notebook location: {notebook_dir}")
print(f"  sam_ml package: {project_root / 'sam_ml'}")
print(f"  Python path includes: {project_root_str in sys.path}")


In [ ]:
# Import necessary libraries
import logging
from pathlib import Path
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)

# Import project configuration for path management
from sam_ml.config import (
    get_project_root,
    get_models_dir,
    get_processed_dataset_path,
)

# Import dataset and model modules
from sam_ml.datasets.eyepacs import load_eyepacs_dual_channel
from sam_ml.modeling.models.dual_channel_model import DualChannelDiabeticRetinopathyModel

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Get project paths
project_root = get_project_root()
models_dir = get_models_dir()
checkpoint_dir = models_dir / "checkpoints"

print("✓ Libraries imported successfully!")
print("✓ Configuration module imported")
print(f"\nProject root: {project_root}")
print(f"Models directory: {models_dir}")
print(f"Checkpoint directory: {checkpoint_dir}")


## Step 3: CUDA/GPU Availability Check

Check if CUDA is available and configure TensorFlow to use GPUs if available. This step is crucial for faster training.

In [ ]:
def check_cuda_availability() -> bool:
    """
    Check if CUDA is available and properly configured.
    
    Returns:
        True if CUDA is available and can be used, False otherwise
    """
    try:
        # Check if TensorFlow can see any GPUs
        gpus = tf.config.list_physical_devices("GPU")
        
        if len(gpus) == 0:
            logger.info("No GPU devices found. Using CPU.")
            return False
        
        # Check if CUDA is built and available
        if not tf.test.is_built_with_cuda():
            logger.warning(
                "GPU devices found but TensorFlow was not built with CUDA support. "
                "Using CPU."
            )
            return False
        
        # Explicitly set visible devices to GPUs
        try:
            tf.config.set_visible_devices(gpus, "GPU")
            logger.info(f"Set visible GPU devices: {[gpu.name for gpu in gpus]}")
        except RuntimeError as e:
            logger.warning(f"Could not set visible GPU devices: {e}")
        
        # Configure GPU memory growth to avoid allocating all memory at once
        for gpu in gpus:
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
                logger.info(f"Configured GPU memory growth: {gpu.name}")
            except RuntimeError as e:
                logger.warning(f"Could not configure GPU memory growth for {gpu.name}: {e}")
        
        # Verify GPU is accessible by checking logical devices
        logical_gpus = tf.config.list_logical_devices("GPU")
        if len(logical_gpus) > 0:
            logger.info(f"CUDA is available. Found {len(gpus)} physical GPU(s) and {len(logical_gpus)} logical GPU(s).")
            for i, logical_gpu in enumerate(logical_gpus):
                logger.info(f"  Logical GPU {i}: {logical_gpu.name}")
        else:
            logger.warning("GPUs detected but no logical devices available. This may indicate a configuration issue.")
            return False
        
        return True
        
    except Exception as e:
        logger.warning(f"Error checking CUDA availability: {e}. Using CPU.")
        return False

# Check CUDA availability
use_cuda = check_cuda_availability()

if use_cuda:
    print("\n✓ CUDA is available! Training will use GPU.")
else:
    print("\n⚠️  CUDA is not available. Training will use CPU (may be slower).")

In [ ]:
# Import necessary libraries
import logging
from pathlib import Path
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)

# Import project configuration for path management
from sam_ml.config import (
    get_project_root,
    get_models_dir,
    get_processed_dataset_path,
)

# Import dataset and model modules
from sam_ml.datasets.eyepacs import load_eyepacs_dual_channel
from sam_ml.modeling.models.dual_channel_model import DualChannelDiabeticRetinopathyModel

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Get project paths
project_root = get_project_root()
models_dir = get_models_dir()
checkpoint_dir = models_dir / "checkpoints"

print("✓ Libraries imported successfully!")
print("✓ Configuration module imported")
print(f"\nProject root: {project_root}")
print(f"Models directory: {models_dir}")
print(f"Checkpoint directory: {checkpoint_dir}")


## Step 3: Training Configuration

Set up all training hyperparameters and configuration options. You can modify these values to experiment with different settings.


In [ ]:
# Dataset configuration
DATASET_NAME = "eyepacs_dataset"
# Use project root path for dataset (works on Paperspace)
BASE_PATH = get_processed_dataset_path(DATASET_NAME)
BATCH_SIZE = 16  # Reduced from 32 - helps with stability and memory, better gradient estimates
IMAGE_SIZE_CLAHE = (299, 299)  # For InceptionV3
IMAGE_SIZE_CECED = (224, 224)  # For VGG-16
LABEL_MODE = "categorical"
SHUFFLE = True
SEED = 42
# Cache strategy: 
# - Training: No cache (large dataset, shuffled each epoch - prefetch is sufficient)
# - Validation/Test: Memory cache (small datasets, reused frequently)
# Set to False to disable caching entirely (saves memory, slightly slower validation)
CACHE = True
PREFETCH = True

# Model configuration
NUM_CLASSES = 5
INPUT_SHAPE = (224, 224, 3)  # Note: This is for reference, actual inputs are dual-channel

# Training configuration
EPOCHS = 50  # Increased from 5 - model needs more time to learn
VALIDATION_FREQ = 1
VERBOSE = 1

# Optimizer hyperparameters
# Lower learning rate for fine-tuning pretrained models (InceptionV3, VGG16)
LEARNING_RATE = 0.0001  # Reduced from 0.001 - better for fine-tuning
BETA_1 = 0.9
BETA_2 = 0.999
EPSILON = 1e-7
WEIGHT_DECAY = 0.0  # Note: Not directly supported by Adam, use kernel_regularizer for weight decay

# Loss and metrics
# Use label smoothing to prevent overconfidence and improve generalization
LABEL_SMOOTHING = 0.1  # Helps with overfitting and improves validation performance
LOSS = "categorical_crossentropy"
METRICS = ["accuracy", "top_k_categorical_accuracy"]  # Add top-k accuracy for better monitoring

# Callback configuration
CHECKPOINT_FILENAME = "best_model.keras"
MONITOR = "val_loss"
MODE = "min"
PATIENCE = 15  # Increased patience - give model more time to improve
RESTORE_BEST_WEIGHTS = True
REDUCE_LR_PATIENCE = 8  # More patience before reducing LR
REDUCE_LR_FACTOR = 0.3  # More aggressive LR reduction (was 0.5)
REDUCE_LR_MIN_LR = 1e-6  # Slightly higher minimum (was 1e-7)

# Class weights for imbalanced dataset (common in medical imaging)
# These weights are calculated from the actual dataset distribution
# to balance the loss contribution from each class
USE_CLASS_WEIGHTS = True  # Enable class weighting
# Balanced weights calculated from dataset: Class 0=73.6%, 1=7.0%, 2=15.0%, 3=2.5%, 4=2.0%
CLASS_WEIGHTS = {
    0: 0.272,   # No DR - 73.6% of dataset, reduced weight
    1: 2.849,   # Mild - 7.0% of dataset, increased weight
    2: 1.335,   # Moderate - 15.0% of dataset, slightly increased
    3: 8.057,   # Severe - 2.5% of dataset, significantly increased
    4: 10.197,  # Proliferative - 2.0% of dataset (rarest), highest weight
}

print("Training Configuration:")
print("=" * 60)
print(f"Dataset: {DATASET_NAME}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Image Sizes: CLAHE={IMAGE_SIZE_CLAHE}, CECED={IMAGE_SIZE_CECED}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Optimizer: Adam (β1={BETA_1}, β2={BETA_2}, ε={EPSILON})")
print(f"Loss: {LOSS}")
print(f"Metrics: {METRICS}")
print(f"CUDA: {use_cuda}")
print(f"Checkpoint: {checkpoint_dir / CHECKPOINT_FILENAME}")
print("=" * 60)


## Step 4: Create Model

Create the dual-channel diabetic retinopathy detection model. This model uses two input channels:
- **CLAHE channel**: Preprocessed images for InceptionV3 (299×299)
- **CECED channel**: Edge-detected images for VGG-16 (224×224)


In [ ]:
# Dataset configuration
DATASET_NAME = "eyepacs_dataset"
BASE_PATH = get_processed_dataset_path(DATASET_NAME)
BATCH_SIZE = 16  # Reduced from 32 - helps with stability and memory, better gradient estimates
IMAGE_SIZE_CLAHE = (299, 299)  # For InceptionV3
IMAGE_SIZE_CECED = (224, 224)  # For VGG-16
LABEL_MODE = "categorical"
SHUFFLE = True
SEED = 42
# Cache strategy: 
# - Training: No cache (large dataset, shuffled each epoch - prefetch is sufficient)
# - Validation/Test: Memory cache (small datasets, reused frequently)
# Set to False to disable caching entirely (saves memory, slightly slower validation)
CACHE = True
PREFETCH = True

# Model configuration
NUM_CLASSES = 5
INPUT_SHAPE = (224, 224, 3)  # Note: This is for reference, actual inputs are dual-channel

# Training configuration
EPOCHS = 5
VALIDATION_FREQ = 1
VERBOSE = 1

# Optimizer hyperparameters
# Lower learning rate for fine-tuning pretrained models (InceptionV3, VGG16)
LEARNING_RATE = 0.0001  # Reduced from 0.001 - better for fine-tuning
BETA_1 = 0.9
BETA_2 = 0.999
EPSILON = 1e-7
WEIGHT_DECAY = 0.0  # Note: Not directly supported by Adam, use kernel_regularizer for weight decay

# Loss and metrics
# Use label smoothing to prevent overconfidence and improve generalization
LABEL_SMOOTHING = 0.1  # Helps with overfitting and improves validation performance
LOSS = "categorical_crossentropy"
METRICS = ["accuracy", "top_k_categorical_accuracy"]  # Add top-k accuracy for better monitoring

# Callback configuration
CHECKPOINT_FILENAME = "best_model.keras"
MONITOR = "val_loss"
MODE = "min"
PATIENCE = 15  # Increased patience - give model more time to improve
RESTORE_BEST_WEIGHTS = True
REDUCE_LR_PATIENCE = 8  # More patience before reducing LR
REDUCE_LR_FACTOR = 0.3  # More aggressive LR reduction (was 0.5)
REDUCE_LR_MIN_LR = 1e-6  # Slightly higher minimum (was 1e-7)

# Class weights for imbalanced dataset (common in medical imaging)
# These weights are calculated from the actual dataset distribution
# to balance the loss contribution from each class
USE_CLASS_WEIGHTS = True  # Enable class weighting
# Balanced weights calculated from dataset: Class 0=73.6%, 1=7.0%, 2=15.0%, 3=2.5%, 4=2.0%
CLASS_WEIGHTS = {
    0: 0.272,   # No DR - 73.6% of dataset, reduced weight
    1: 2.849,   # Mild - 7.0% of dataset, increased weight
    2: 1.335,   # Moderate - 15.0% of dataset, slightly increased
    3: 8.057,   # Severe - 2.5% of dataset, significantly increased
    4: 10.197,  # Proliferative - 2.0% of dataset (rarest), highest weight
}

print("Training Configuration:")
print("=" * 60)
def create_model(
    num_classes: int = 5,
    input_shape: Tuple[int, int, int] = (224, 224, 3),
) -> DualChannelDiabeticRetinopathyModel:
    """
    Create a dual-channel diabetic retinopathy detection model.
    
    Args:
        num_classes: Number of output classes (default: 5)
        input_shape: Shape of input images (height, width, channels) (default: (224, 224, 3))
        
    Returns:
        Model instance (not compiled)
    """
    model = DualChannelDiabeticRetinopathyModel(
        num_classes=num_classes,
        input_shape=input_shape,
    )
    
    return model

# Create the model
logger.info("Creating dual-channel model...")
model = create_model(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE)

print("✓ Model created successfully!")
print(f"\nModel name: {model.name}")
print(f"Model configuration:")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  CLAHE channel: Input resized to 299×299 for InceptionV3")
print(f"  CECED channel: Input size 224×224 for VGG-16")
print(f"\nModel summary:")
model.summary()


## Step 4: Create Model

Create the dual-channel diabetic retinopathy detection model. This model uses two input channels:
- **CLAHE channel**: Preprocessed images for InceptionV3 (299×299)
- **CECED channel**: Edge-detected images for VGG-16 (224×224)


In [ ]:
def create_model(
    num_classes: int = 5,
    input_shape: Tuple[int, int, int] = (224, 224, 3),
) -> DualChannelDiabeticRetinopathyModel:
    """
    Create a dual-channel diabetic retinopathy detection model.
    
    Args:
        num_classes: Number of output classes (default: 5)
        input_shape: Shape of input images (height, width, channels) (default: (224, 224, 3))
        
    Returns:
        Model instance (not compiled)
    """
    model = DualChannelDiabeticRetinopathyModel(
        num_classes=num_classes,
        input_shape=input_shape,
    )
    
    return model

# Create the model
logger.info("Creating dual-channel model...")
model = create_model(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE)

print(f"Model configuration:")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  CLAHE channel: Input resized to 299×299 for InceptionV3")
print(f"  CECED channel: Input size 224×224 for VGG-16")
model.summary()


## Step 5: Load Datasets

Load the EyePACS datasets for training, validation, and testing. The datasets are loaded as dual-channel datasets with CLAHE and CECED preprocessed images.


In [ ]:
def load_dataset(
    base_path: Optional[Path] = None,
    dataset_name: str = "eyepacs_dataset",
    batch_size: int = 32,
    image_size_clahe: Tuple[int, int] = (299, 299),
    image_size_ceced: Tuple[int, int] = (224, 224),
    label_mode: str = "categorical",
    shuffle: bool = True,
    seed: Optional[int] = 42,
    cache: bool = True,
    prefetch: bool = True,
):
    """
    Load EyePACS dataset for dual-channel model training.
    
    Returns:
        Tuple of (train_dataset, val_dataset, test_dataset)
    """
    train, val, test = load_eyepacs_dual_channel(
        base_path=base_path,
        dataset_name=dataset_name,
        batch_size=batch_size,
        image_size_clahe=image_size_clahe,
        image_size_ceced=image_size_ceced,
        label_mode=label_mode,
        shuffle=shuffle,
        seed=seed,
        cache=cache,
        prefetch=prefetch,
    )
    
    return train, val, test

# Load datasets
logger.info("Loading EyePACS datasets...")
try:
    train_dataset, val_dataset, test_dataset = load_dataset(
        base_path=BASE_PATH,
        dataset_name=DATASET_NAME,
        batch_size=BATCH_SIZE,
        image_size_clahe=IMAGE_SIZE_CLAHE,
        image_size_ceced=IMAGE_SIZE_CECED,
        label_mode=LABEL_MODE,
        shuffle=SHUFFLE,
        seed=SEED,
        cache=CACHE,
        prefetch=PREFETCH,
    )
    
    print("✓ Datasets loaded successfully!")
    print(f"\nDataset information:")
    print(f"  Training batches: {len(train_dataset)}")
    print(f"  Validation batches: {len(val_dataset)}")
    print(f"  Test batches: {len(test_dataset)}")
    
    # Get a sample batch to verify shapes
    # The dataset returns ((clahe, ceced), labels) structure
    sample_dataset = train_dataset.take(1)
    sample_batch = next(iter(sample_dataset))
    (clahe_batch, ceced_batch), labels = sample_batch
    print(f"\nSample batch shapes:")
    print(f"  CLAHE batch: {clahe_batch.shape}")
    print(f"  CECED batch: {ceced_batch.shape}")
    print(f"  Labels: {labels.shape}")
    
    # Calculate class distribution for better class weights
    if USE_CLASS_WEIGHTS:
        print(f"\nCalculating class distribution from training set...")
        class_counts = {i: 0 for i in range(NUM_CLASSES)}
        total_samples = 0
        
        # Count samples per class - iterate through ALL batches to get accurate distribution
        # The dataset returns ((clahe, ceced), labels) structure
        # Note: This may take a moment but ensures accurate class distribution
        print(f"  Iterating through training batches (this may take a moment)...")
        batch_count = 0
        for batch in train_dataset:
            (_, _), batch_labels = batch  # Unpack correctly: inputs is (clahe, ceced), then labels
            # Convert one-hot to class indices
            class_indices = tf.argmax(batch_labels, axis=1)
            for class_idx in class_indices.numpy():
                class_counts[int(class_idx)] += 1
                total_samples += 1
            batch_count += 1
            if batch_count % 100 == 0:
                print(f"    Processed {batch_count} batches ({total_samples} samples)...")
        
        print(f"\n  Total samples analyzed: {total_samples}")
        print(f"  Class distribution:")
        for class_idx, count in class_counts.items():
            percentage = (count / total_samples * 100) if total_samples > 0 else 0
            print(f"    Class {class_idx}: {count} samples ({percentage:.1f}%)")
        
        # Calculate balanced class weights (sklearn style)
        # weight = n_samples / (n_classes * np.bincount(y))
        if total_samples > 0:
            balanced_weights = {}
            for class_idx in range(NUM_CLASSES):
                if class_counts[class_idx] > 0:
                    balanced_weights[class_idx] = total_samples / (NUM_CLASSES * class_counts[class_idx])
                else:
                    balanced_weights[class_idx] = 1.0
            
            print(f"\n  Suggested balanced class weights:")
            for class_idx, weight in balanced_weights.items():
                print(f"    Class {class_idx}: {weight:.3f}")
            print(f"\n  Note: Update CLASS_WEIGHTS in config cell if you want to use these values.")
    
except FileNotFoundError as e:
    print(f"⚠️  Dataset not found!")
    print(f"\nExpected dataset path: {BASE_PATH}")
    print(f"\nTo generate the processed dataset, run:")
    print(f"  python -m sam_ml.preprocessing {DATASET_NAME}")
    print(f"\nThis will download the dataset from Hugging Face and process it.")
    raise
except Exception as e:
    print(f"ERROR loading datasets: {e}")
    import traceback
    traceback.print_exc()
    raise


## Step 6: Create Optimizer

Create the Adam optimizer with the specified hyperparameters.


In [ ]:
def create_adam_optimizer(
    learning_rate: float = 0.001,
    beta_1: float = 0.9,
    beta_2: float = 0.999,
    epsilon: float = 1e-7,
    weight_decay: float = 0.0,
) -> keras.optimizers.Adam:
    """
    Create an Adam optimizer with specified hyperparameters.
    
    Args:
        learning_rate: Learning rate (default: 0.001)
        beta_1: Exponential decay rate for the first moment estimates (default: 0.9)
        beta_2: Exponential decay rate for the second moment estimates (default: 0.999)
        epsilon: Small constant for numerical stability (default: 1e-7)
        weight_decay: Weight decay coefficient (default: 0.0)
                     Note: Weight decay in TensorFlow/Keras is typically handled via
                     kernel_regularizer in layers.
        
    Returns:
        Configured Adam optimizer
    """
    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate,
        beta_1=beta_1,
        beta_2=beta_2,
        epsilon=epsilon,
    )
    
    if weight_decay > 0.0:
        logger.warning(
            f"Weight decay ({weight_decay}) is specified but not directly supported "
            "by Adam optimizer. Consider using kernel_regularizer in model layers "
            "or an optimizer wrapper for weight decay."
        )
    
    return optimizer

# Create optimizer
logger.info("Creating Adam optimizer...")
optimizer = create_adam_optimizer(
    learning_rate=LEARNING_RATE,
    beta_1=BETA_1,
    beta_2=BETA_2,
    epsilon=EPSILON,
    weight_decay=WEIGHT_DECAY,
)

print("✓ Optimizer created successfully!")
print(f"\nOptimizer: {optimizer}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Beta 1: {BETA_1}")
print(f"  Beta 2: {BETA_2}")
print(f"  Epsilon: {EPSILON}")


## Step 7: Create Callbacks

Set up training callbacks for:
- **Model Checkpointing**: Save the best model based on validation loss
- **Early Stopping**: Stop training if validation loss doesn't improve
- **Learning Rate Reduction**: Reduce learning rate when validation loss plateaus


In [ ]:
def create_callbacks(
    checkpoint_dir: Optional[Path] = None,
    checkpoint_filename: str = "best_model.keras",
    monitor: str = "val_loss",
    mode: str = "min",
    patience: int = 10,
    restore_best_weights: bool = True,
    reduce_lr_patience: int = 5,
    reduce_lr_factor: float = 0.5,
    reduce_lr_min_lr: float = 1e-7,
    verbose: int = 1,
):
    """
    Create training callbacks for model training.
    
    Returns:
        List of Keras callbacks
    """
    callbacks = []
    
    # Model checkpoint callback
    if checkpoint_dir is not None:
        checkpoint_dir = Path(checkpoint_dir)
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        checkpoint_path = checkpoint_dir / checkpoint_filename
        
        checkpoint_callback = ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor=monitor,
            mode=mode,
            save_best_only=True,
            save_weights_only=False,
            verbose=verbose,
        )
        callbacks.append(checkpoint_callback)
        logger.info(f"Model checkpoint will be saved to: {checkpoint_path}")
    
    # Early stopping callback
    early_stopping = EarlyStopping(
        monitor=monitor,
        mode=mode,
        patience=patience,
        restore_best_weights=restore_best_weights,
        verbose=verbose,
    )
    callbacks.append(early_stopping)
    
    # Learning rate reduction callback
    reduce_lr = ReduceLROnPlateau(
        monitor=monitor,
        mode=mode,
        factor=reduce_lr_factor,
        patience=reduce_lr_patience,
        min_lr=reduce_lr_min_lr,
        verbose=verbose,
    )
    callbacks.append(reduce_lr)
    
    return callbacks

# Create callbacks
logger.info("Creating training callbacks...")
callbacks = create_callbacks(
    checkpoint_dir=checkpoint_dir,
    checkpoint_filename=CHECKPOINT_FILENAME,
    monitor=MONITOR,
    mode=MODE,
    patience=PATIENCE,
    restore_best_weights=RESTORE_BEST_WEIGHTS,
    reduce_lr_patience=REDUCE_LR_PATIENCE,
    reduce_lr_factor=REDUCE_LR_FACTOR,
    reduce_lr_min_lr=REDUCE_LR_MIN_LR,
    verbose=VERBOSE,
)

print("✓ Callbacks created successfully!")
print(f"\nCallbacks ({len(callbacks)}):")
for i, callback in enumerate(callbacks, 1):
    print(f"  {i}. {callback.__class__.__name__}")
    if isinstance(callback, ModelCheckpoint):
        print(f"     Filepath: {callback.filepath}")
    elif isinstance(callback, EarlyStopping):
        print(f"     Monitor: {callback.monitor}, Patience: {callback.patience}")
    elif isinstance(callback, ReduceLROnPlateau):
        print(f"     Monitor: {callback.monitor}, Factor: {callback.factor}, Patience: {callback.patience}")


## Step 8: Compile Model

Compile the model with the optimizer, loss function, and metrics. This prepares the model for training.


In [ ]:
# Compile model
logger.info("Compiling model...")
# Use label smoothing in loss to improve generalization
loss_with_smoothing = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
model.compile(
    optimizer=optimizer,
    loss=loss_with_smoothing,
    metrics=METRICS,
)

print("✓ Model compiled successfully!")
print(f"\nCompilation details:")
print(f"  Optimizer: {model.optimizer}")
print(f"  Loss: {model.loss}")
print(f"  Metrics: {model.metrics_names}")

# Verify GPU usage after compilation
if use_cuda:
    try:
        logical_gpus = tf.config.list_logical_devices("GPU")
        physical_gpus = tf.config.list_physical_devices("GPU")
        
        if len(logical_gpus) > 0:
            logger.info(f"GPU devices available for training:")
            for i, gpu in enumerate(logical_gpus):
                logger.info(f"  - {gpu.name}")
            logger.info(f"Total: {len(physical_gpus)} physical GPU(s), {len(logical_gpus)} logical GPU(s)")
            
            # Test GPU placement with a simple operation
            with tf.device("/GPU:0"):
                test_tensor = tf.constant([1.0, 2.0, 3.0])
                test_result = tf.reduce_sum(test_tensor)
                device_used = test_result.device
                logger.info(f"GPU placement test: Operations will run on {device_used}")
        else:
            logger.warning("GPU was requested but no logical GPU devices found. Model may run on CPU.")
    except Exception as e:
        logger.warning(f"Could not verify GPU device placement: {e}")


## Step 9: Train Model

Train the model on the training dataset with validation on the validation dataset. The training will use the configured callbacks for checkpointing, early stopping, and learning rate reduction.


In [ ]:
# Log training configuration
logger.info("=" * 60)
logger.info("Training Configuration:")
logger.info(f"  Model: {model.name}")
logger.info(f"  Epochs: {EPOCHS}")
logger.info(f"  Optimizer: {optimizer}")
if isinstance(optimizer, keras.optimizers.Adam):
    logger.info(f"    Learning Rate: {optimizer.learning_rate.numpy() if hasattr(optimizer.learning_rate, 'numpy') else optimizer.learning_rate}")
    logger.info(f"    Beta 1: {optimizer.beta_1.numpy() if hasattr(optimizer.beta_1, 'numpy') else optimizer.beta_1}")
    logger.info(f"    Beta 2: {optimizer.beta_2.numpy() if hasattr(optimizer.beta_2, 'numpy') else optimizer.beta_2}")
    logger.info(f"    Epsilon: {optimizer.epsilon}")
logger.info(f"  Loss: {LOSS}")
logger.info(f"  Metrics: {METRICS}")
logger.info(f"  CUDA: {use_cuda}")
logger.info(f"  Callbacks: {len(callbacks)}")
logger.info("=" * 60)

# Prepare class weights if enabled
class_weight_dict = None
if USE_CLASS_WEIGHTS:
    class_weight_dict = CLASS_WEIGHTS
    logger.info(f"Using class weights: {class_weight_dict}")

# Train model
logger.info("Starting training...")

# Use GPU device scope if CUDA is available
if use_cuda:
    logical_gpus = tf.config.list_logical_devices("GPU")
    if len(logical_gpus) > 0:
        logger.info(f"Training with GPU device scope: {logical_gpus[0].name}")
        with tf.device(logical_gpus[0].name):
            history = model.fit(
                train_dataset,
                validation_data=val_dataset,
                epochs=EPOCHS,
                callbacks=callbacks,
                validation_freq=VALIDATION_FREQ,
                verbose=VERBOSE,
                class_weight=class_weight_dict,
            )
    else:
        logger.warning("CUDA requested but no logical GPU devices found. Training on CPU.")
        history = model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=EPOCHS,
            callbacks=callbacks,
            validation_freq=VALIDATION_FREQ,
            verbose=VERBOSE,
            class_weight=class_weight_dict,
        )
else:
    # Train on CPU
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=callbacks,
        validation_freq=VALIDATION_FREQ,
        verbose=VERBOSE,
        class_weight=class_weight_dict,
    )

logger.info("Training completed!")
print("\n✓ Training completed successfully!")


## Step 10: Evaluate on Test Set

Evaluate the trained model on the test dataset to get final performance metrics.


In [ ]:
# Evaluate on test set
logger.info("Evaluating on test set...")

if use_cuda:
    logical_gpus = tf.config.list_logical_devices("GPU")
    if len(logical_gpus) > 0:
        with tf.device(logical_gpus[0].name):
            test_results = model.evaluate(test_dataset, verbose=VERBOSE)
    else:
        test_results = model.evaluate(test_dataset, verbose=VERBOSE)
else:
    test_results = model.evaluate(test_dataset, verbose=VERBOSE)

# Display results
test_metrics = dict(zip(model.metrics_names, test_results))
logger.info(f"Test results: {test_metrics}")

print("\n✓ Test evaluation completed!")
print(f"\nTest Results:")
for metric_name, metric_value in test_metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")


## Step 11: Visualize Training History

Plot the training and validation metrics over epochs to visualize the training progress.


In [ ]:
# Plot training history
def plot_training_history(history):
    """
    Plot training and validation metrics.
    
    Args:
        history: Keras training history object
    """
    # Get available metrics
    metrics = list(history.history.keys())
    
    # Separate loss and other metrics
    loss_metrics = [m for m in metrics if 'loss' in m.lower()]
    other_metrics = [m for m in metrics if 'loss' not in m.lower()]
    
    # Determine number of subplots
    n_plots = len(loss_metrics) + len(other_metrics)
    if n_plots == 0:
        print("No metrics found in history.")
        return
    
    # Create subplots
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]
    
    plot_idx = 0
    
    # Plot loss metrics
    for metric in loss_metrics:
        ax = axes[plot_idx]
        ax.plot(history.history[metric], label=f'Training {metric}')
        if f'val_{metric}' in history.history:
            ax.plot(history.history[f'val_{metric}'], label=f'Validation {metric}')
        ax.set_title(f'{metric.capitalize()}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric.capitalize())
        ax.legend()
        ax.grid(True, alpha=0.3)
        plot_idx += 1
    
    # Plot other metrics
    for metric in other_metrics:
        ax = axes[plot_idx]
        ax.plot(history.history[metric], label=f'Training {metric}')
        if f'val_{metric}' in history.history:
            ax.plot(history.history[f'val_{metric}'], label=f'Validation {metric}')
        ax.set_title(f'{metric.capitalize()}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric.capitalize())
        ax.legend()
        ax.grid(True, alpha=0.3)
        plot_idx += 1
    
    plt.tight_layout()
    plt.show()

# Plot training history
if history is not None:
    print("Plotting training history...")
    plot_training_history(history)
    print("✓ Training history plotted!")
else:
    print("⚠️  No training history available to plot.")


## Summary

This notebook has completed the full training process:

1. ✓ **Setup and Configuration**: Libraries imported and paths configured
2. ✓ **CUDA/GPU Check**: GPU availability verified
3. ✓ **Model Creation**: Dual-channel model created
4. ✓ **Dataset Loading**: Training, validation, and test datasets loaded
5. ✓ **Optimizer Setup**: Adam optimizer configured
6. ✓ **Callbacks Setup**: Checkpointing, early stopping, and LR reduction configured
7. ✓ **Model Compilation**: Model compiled with optimizer, loss, and metrics
8. ✓ **Training**: Model trained on training dataset
9. ✓ **Evaluation**: Model evaluated on test dataset
10. ✓ **Visualization**: Training history plotted

### Model Checkpoint

The best model (based on validation loss) has been saved to:
- **Path**: `models/checkpoints/best_model.keras`

You can load this model later using:
```python
from tensorflow import keras
model = keras.models.load_model("models/checkpoints/best_model.keras")
```

### Next Steps

- Experiment with different hyperparameters (learning rate, batch size, etc.)
- Try different model architectures
- Perform hyperparameter tuning
- Analyze model predictions on specific samples
- Export the model for deployment
